##ESG Controversy Prediction

Dataset  : 10 Indian banks × 4 years (2022–2025) = 40 rows

Target   : Actual_ESG_Controversy_Level  (0 = no controversy, 1 = controversy)

Strategy : Lagged prediction — year-t features → year-(t+1) controversy

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.inspection      import permutation_importance
from sklearn.metrics         import (accuracy_score, precision_score,
                                     recall_score, f1_score, roc_auc_score,
                                     confusion_matrix, ConfusionMatrixDisplay,
                                     RocCurveDisplay, PrecisionRecallDisplay)
from xgboost import XGBClassifier

print("=" * 70)
print("   ESG CONTROVERSY PREDICTION PIPELINE")
print("=" * 70)

   ESG CONTROVERSY PREDICTION PIPELINE


# Part 1: Load and clean data


In [ ]:
print("\n[PART 1] Loading & Cleaning Data …")

df = pd.read_excel("/content/final database for esg.xlsx")
print(f"  Raw shape: {df.shape}")


[PART 1] Loading & Cleaning Data …
  Raw shape: (40, 28)


##1.a:  rename messy column names


In [ ]:
rename_map = {
    "Total Assets (₹ Cr)"  : "TotalAssets_Cr",
    "Equity (₹ Cr) "       : "Equity_Cr",
    "Total Loans (₹ Cr)"   : "TotalLoans_Cr",
    "Net Profit (₹ Cr) "   : "NetProfit_Cr",
    "NPL/Loans % "         : "NPL_Loans_pct",
    "Capad (E/A) "         : "Capad",
    "NPL/Assets % "        : "NPL_Assets_pct",
    "Loans/Assets "        : "Loans_Assets",
    "Earnings (NI/A) %"    : "Earnings_NIA_pct",
    "Year-end Px (₹)"      : "YearEndPrice",
    "Employee Count"       : "EmployeeCount",
}
df.rename(columns=rename_map, inplace=True)

## 1b. Drop duplicate / redundant columns ─────────────────────────────────

*   Ticker_x       → not a predictor
*   NetIncome      → duplicate of NetProfit_Cr (different scale, same concept)
*   TotalAssets    → duplicate of TotalAssets_Cr
*   OperatingIncome → identical to TotalRevenue in this dataset
*   Shares, YearEndPrice, AverageMarketCapCrore → leakage / noise for small set










In [ ]:
drop_cols = [
    "Ticker_x", "NetIncome", "TotalAssets", "Shares (Cr)", "YearEndPrice",
    "AverageMarketCapCrore"
]
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

In [ ]:
# ── 1d. Basic missing-value report ─────────────────────────────────────────
missing = df.isnull().sum()
if missing.any():
    print("  Missing values detected:")
    print(missing[missing > 0].to_string())
else:
    print("  No missing values.")

  No missing values.


In [ ]:
# Fill any remaining NaNs with column median (conservative imputation)
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].apply(lambda col: col.fillna(col.median()))

print(f"  Clean shape: {df.shape}")
print(f"  Columns kept: {list(df.columns)}")
print(f"\n  Target distribution:\n{df['Actual_ESG_Controversy_Level'].value_counts().to_string()}")

  Clean shape: (40, 23)
  Columns kept: ['Bank', 'Year', 'TotalAssets_Cr', 'Equity_Cr', 'TotalLoans_Cr', 'NetProfit_Cr', 'NPL_Loans_pct', 'Formula_Fin_Lev', 'Capad', 'NPL_Assets_pct', 'Loans_Assets', 'Earnings_NIA_pct', 'MTBV', 'ROE', 'ROA', 'PE', 'CostToIncomeRatio', 'TotalRevenue', 'Negative_ESG_News_Count', 'Avg_BM25_Score', 'Avg_FinBERT_Confidence', 'EmployeeCount', 'Actual_ESG_Controversy_Level']

  Target distribution:
Actual_ESG_Controversy_Level
0    22
1    18


# PART 2 — FEATURE ENGINEERING


In [ ]:
# Sort so lag operations are correct
df.sort_values(["Bank", "Year"], inplace=True)
df.reset_index(drop=True, inplace=True)

# ── Engineered features (within same year, per bank) ───────────────────────

# 1. Negative News Intensity  = volume × relevance
df["NegNewsIntensity"] = (
    df["Negative_ESG_News_Count"] * df["Avg_BM25_Score"]
)

# 2. Sentiment Risk Score  = volume × FinBERT confidence
df["SentimentRiskScore"] = (
    df["Negative_ESG_News_Count"] * df["Avg_FinBERT_Confidence"]
)

# 3. Controversy Momentum  = ΔNegNews (vs. prior year, per bank)
df["ControversyMomentum"] = df.groupby("Bank")["Negative_ESG_News_Count"].diff().fillna(0)

# 4. ROA Stability  = ΔROA (vs. prior year, per bank)
df["ROA_Stability"] = df.groupby("Bank")["ROA"].diff().fillna(0)

# 5. Governance Stress  = CostToIncomeRatio × NegNewsCount
df["GovernanceStress"] = (
    df["CostToIncomeRatio"] * df["Negative_ESG_News_Count"]
)

print("  Engineered: NegNewsIntensity, SentimentRiskScore, ControversyMomentum,")
print("              ROA_Stability, GovernanceStress")


  Engineered: NegNewsIntensity, SentimentRiskScore, ControversyMomentum,
              ROA_Stability, GovernanceStress


# PART 3 — CREATE LAGGED DATASET  (year-t features → year-(t+1) controversy)


In [ ]:
print("\n[PART 3] Creating Lagged Dataset (t → t+1) …")

# For each bank, shift controversy label back by 1 year so that
#   features of year t predict controversy of year t+1
df["Target_NextYear"] = df.groupby("Bank")["Actual_ESG_Controversy_Level"].shift(-1)

# Keep only rows where we have a valid next-year target
#   (the last year per bank, 2025, gets dropped — no 2026 label)
df_lagged = df.dropna(subset=["Target_NextYear"]).copy()
df_lagged["Target_NextYear"] = df_lagged["Target_NextYear"].astype(int)

print(f"  Lagged dataset shape: {df_lagged.shape}")
print(f"  Years used as features: {sorted(df_lagged['Year'].unique())}")
print(f"  Target distribution:\n{df_lagged['Target_NextYear'].value_counts().to_string()}")




[PART 3] Creating Lagged Dataset (t → t+1) …
  Lagged dataset shape: (30, 29)
  Years used as features: [np.int64(2022), np.int64(2023), np.int64(2024)]
  Target distribution:
Target_NextYear
1    17
0    13


# PART 4 — EXPLORATORY DATA ANALYSIS (EDA)


In [ ]:
print("\n[PART 4] EDA …")

# ── Define feature sets ────────────────────────────────────────────────────
FINANCIAL_FEATURES = [
    "ROE", "ROA", "NPL_Loans_pct", "Formula_Fin_Lev", "Capad",
    "NPL_Assets_pct", "Loans_Assets", "CostToIncomeRatio",
    "TotalRevenue", "NetProfit_Cr", "MTBV", "PE", "EmployeeCount",
    "ROA_Stability"
]
NLP_FEATURES = [
    "Negative_ESG_News_Count", "Avg_BM25_Score", "Avg_FinBERT_Confidence",
    "NegNewsIntensity", "SentimentRiskScore", "ControversyMomentum",
    "GovernanceStress"
]
ALL_FEATURES = FINANCIAL_FEATURES + NLP_FEATURES

# ── 4a. Descriptive statistics ──────────────────────────────────────────────
print("\n  Descriptive Statistics (lagged dataset):")
desc = df_lagged[ALL_FEATURES].describe().T[["mean", "std", "min", "max"]]
print(desc.round(4).to_string())


[PART 4] EDA …

  Descriptive Statistics (lagged dataset):
                                 mean           std           min           max
ROE                      1.162000e-01  4.770000e-02  1.810000e-02  1.728000e-01
ROA                      1.170000e-02  6.600000e-03  2.100000e-03  2.410000e-02
NPL_Loans_pct            3.783700e+00  3.136500e+00  1.120000e+00  1.393000e+01
Formula_Fin_Lev          1.154660e+01  3.983300e+00  5.548800e+00  1.778300e+01
Capad                    9.780000e-02  3.550000e-02  5.620000e-02  1.802000e-01
NPL_Assets_pct           2.162800e+00  1.751300e+00  7.093000e-01  7.936500e+00
Loans_Assets             5.796000e-01  5.590000e-02  4.901000e-01  6.696000e-01
CostToIncomeRatio        5.506000e-01  1.705000e-01  1.904000e-01  9.138000e-01
TotalRevenue             8.779480e+11  7.928003e+11  8.415000e+10  3.236135e+12
NetProfit_Cr             2.106783e+04  1.963293e+04  7.170000e+02  6.954300e+04
MTBV                     1.780600e+00  9.175000e-01  4.00500

In [ ]:
# ── 4b. Correlation heatmap (all features) ─────────────────────────────────
corr_data = df_lagged[ALL_FEATURES + ["Target_NextYear"]]
corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f",
    cmap="RdYlGn", center=0, linewidths=0.5,
    annot_kws={"size": 7}, ax=ax
)
ax.set_title("Feature Correlation Heatmap\n(ESG Lagged Dataset)", fontsize=14, pad=12)
plt.tight_layout()
plt.savefig("plot_1_correlation_heatmap.png", dpi=150)
plt.close()
print("\n  Saved: plot_1_correlation_heatmap.png")


  Saved: plot_1_correlation_heatmap.png


In [ ]:
# ── 4c. Drop highly correlated features (|r| > 0.85) ──────────────────────
def drop_high_corr(df_in, features, threshold=0.85):
    """Returns list of features after removing highly correlated ones."""
    corr = df_in[features].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    kept = [f for f in features if f not in to_drop]
    if to_drop:
        print(f"\n  Dropped (|r|>0.85): {to_drop}")
    else:
        print("\n  No features dropped for multicollinearity.")
    return kept

FIN_SELECTED  = drop_high_corr(df_lagged, FINANCIAL_FEATURES)
ALL_SELECTED  = drop_high_corr(df_lagged, ALL_FEATURES)

print(f"\n  Financial features after selection : {len(FIN_SELECTED)}  → {FIN_SELECTED}")
print(f"  All features after selection       : {len(ALL_SELECTED)}  → {ALL_SELECTED}")


  Dropped (|r|>0.85): ['Capad', 'NPL_Assets_pct', 'NetProfit_Cr', 'EmployeeCount']

  Dropped (|r|>0.85): ['Capad', 'NPL_Assets_pct', 'NetProfit_Cr', 'EmployeeCount', 'NegNewsIntensity', 'SentimentRiskScore', 'GovernanceStress']

  Financial features after selection : 10  → ['ROE', 'ROA', 'NPL_Loans_pct', 'Formula_Fin_Lev', 'Loans_Assets', 'CostToIncomeRatio', 'TotalRevenue', 'MTBV', 'PE', 'ROA_Stability']
  All features after selection       : 14  → ['ROE', 'ROA', 'NPL_Loans_pct', 'Formula_Fin_Lev', 'Loans_Assets', 'CostToIncomeRatio', 'TotalRevenue', 'MTBV', 'PE', 'ROA_Stability', 'Negative_ESG_News_Count', 'Avg_BM25_Score', 'Avg_FinBERT_Confidence', 'ControversyMomentum']


In [ ]:
# ── 4d. Class balance bar chart ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
counts = df_lagged["Target_NextYear"].value_counts().sort_index()
bars = ax.bar(["No Controversy (0)", "Controversy (1)"],
              counts.values, color=["#4CAF50", "#F44336"], width=0.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(val), ha="center", fontsize=12, fontweight="bold")
ax.set_title("Target Class Distribution (Lagged Dataset)", fontsize=12)
ax.set_ylabel("Count")
ax.set_ylim(0, max(counts.values) + 2)
plt.tight_layout()
plt.savefig("plot_2_class_balance.png", dpi=150)
plt.close()
print("  Saved: plot_2_class_balance.png")

  Saved: plot_2_class_balance.png


# PART 5 — MODELLING HELPERS


In [ ]:
print("\n[PART 5] Setting up Models & CV …")

TARGET = "Target_NextYear"
y = df_lagged[TARGET].values

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

MODELS = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, class_weight="balanced",
        max_depth=4, random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
        use_label_encoder=False, eval_metric="logloss",
        random_state=42, verbosity=0
    ),
}

SCORING = ["accuracy", "precision", "recall", "f1", "roc_auc"]



[PART 5] Setting up Models & CV …


In [ ]:
def run_cv(X_arr, y_arr, scaler=True):
    """Run all three models with 5-fold CV; return results dict."""
    if scaler:
        sc = StandardScaler()
        X_arr = sc.fit_transform(X_arr)
    results = {}
    for name, clf in MODELS.items():
        cv_res = cross_validate(clf, X_arr, y_arr, cv=CV,
                                scoring=SCORING, return_train_score=False)
        results[name] = {
            "Accuracy" : cv_res["test_accuracy"].mean(),
            "Precision": cv_res["test_precision"].mean(),
            "Recall"   : cv_res["test_recall"].mean(),
            "F1"       : cv_res["test_f1"].mean(),
            "ROC-AUC"  : cv_res["test_roc_auc"].mean(),
        }
    return results

def results_table(res_dict, title):
    df_res = pd.DataFrame(res_dict).T.round(4)
    print(f"\n  {title}")
    print("  " + "-" * 60)
    print(df_res.to_string())
    return df_res

# PART 6 — CORE EXPERIMENT: Financial ONLY  vs  Financial + NLP


In [ ]:
print("\n[PART 6] Core Experiment …")

X_fin = df_lagged[FIN_SELECTED].values
X_all = df_lagged[ALL_SELECTED].values

print("\n  ── EXPERIMENT A: Financial Features Only ──")
res_fin = run_cv(X_fin, y)
df_fin  = results_table(res_fin, "Financial Only — 5-Fold CV")

print("\n  ── EXPERIMENT B: Financial + NLP Features ──")
res_all = run_cv(X_all, y)
df_all  = results_table(res_all, "Financial + NLP — 5-Fold CV")

# ── Side-by-side comparison bar chart ──────────────────────────────────────
metrics = ["Accuracy", "F1", "ROC-AUC"]
models  = list(MODELS.keys())
x_pos   = np.arange(len(models))
width   = 0.18

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Experiment A vs B: Financial Only vs Financial + NLP\n(5-Fold Stratified CV)", fontsize=13)

for i, metric in enumerate(metrics):
    ax = axes[i]
    vals_fin = [res_fin[m][metric] for m in models]
    vals_all = [res_all[m][metric] for m in models]
    b1 = ax.bar(x_pos - width/2, vals_fin, width, label="Fin Only",
                color="#5B9BD5", edgecolor="white")
    b2 = ax.bar(x_pos + width/2, vals_all, width, label="Fin + NLP",
                color="#ED7D31", edgecolor="white")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(["LR", "RF", "XGB"], fontsize=10)
    ax.set_ylim(0, 1.15)
    ax.set_title(metric, fontsize=11)
    ax.set_ylabel("Score")
    ax.legend(fontsize=8)
    for b in [*b1, *b2]:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
                f"{b.get_height():.2f}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("plot_3_experiment_comparison.png", dpi=150)
plt.close()
print("\n  Saved: plot_3_experiment_comparison.png")



[PART 6] Core Experiment …

  ── EXPERIMENT A: Financial Features Only ──

  Financial Only — 5-Fold CV
  ------------------------------------------------------------
                     Accuracy  Precision  Recall      F1  ROC-AUC
Logistic Regression    0.4000     0.3533  0.3500  0.3467   0.4917
Random Forest          0.6333     0.7200  0.6500  0.6548   0.6806
XGBoost                0.6667     0.7667  0.6667  0.6762   0.7028

  ── EXPERIMENT B: Financial + NLP Features ──

  Financial + NLP — 5-Fold CV
  ------------------------------------------------------------
                     Accuracy  Precision  Recall      F1  ROC-AUC
Logistic Regression    0.4333     0.5533  0.4000  0.4267   0.3972
Random Forest          0.6667     0.7833  0.6667  0.6857   0.5861
XGBoost                0.6333     0.6667  0.6333  0.6395   0.6611

  Saved: plot_3_experiment_comparison.png


# PART 7 — FEATURE IMPORTANCE & EXPLAINABILITY


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)
feature_names = ALL_SELECTED

# ── 7a. Random Forest built-in importance ──────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=300, class_weight="balanced",
    max_depth=4, random_state=42
)
rf.fit(X_scaled, y)
rf_imp = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)

# ── 7b. Permutation importance (more reliable) ──────────────────────────────
perm = permutation_importance(rf, X_scaled, y,
                               n_repeats=30, random_state=42, n_jobs=-1)
perm_imp = pd.Series(perm.importances_mean, index=feature_names).sort_values(ascending=False)

# ── 7c. Plot both side by side ──────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# RF built-in
colors_rf = ["#ED7D31" if "ESG" in f or "BM25" in f or "Bert" in f.replace("FinBERT","Bert")
             or f in NLP_FEATURES else "#5B9BD5"
             for f in rf_imp.index]
ax1.barh(rf_imp.index[::-1], rf_imp.values[::-1], color=colors_rf[::-1])
ax1.set_title("Random Forest Built-in\nFeature Importance", fontsize=12)
ax1.set_xlabel("Mean Decrease in Impurity")
ax1.axvline(0, color="black", linewidth=0.5)

# Permutation
colors_pi = ["#ED7D31" if f in NLP_FEATURES else "#5B9BD5" for f in perm_imp.index]
ax2.barh(perm_imp.index[::-1], perm_imp.values[::-1], color=colors_pi[::-1])
ax2.set_title("Permutation Importance\n(More Reliable)", fontsize=12)
ax2.set_xlabel("Mean Accuracy Drop on Shuffle")
ax2.axvline(0, color="black", linewidth=0.5)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#ED7D31", label="NLP / ESG Signal"),
    Patch(facecolor="#5B9BD5", label="Financial Feature")
]
fig.legend(handles=legend_elements, loc="lower center",
           ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Feature Importance (Financial + NLP Model | Random Forest)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("plot_4_feature_importance.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: plot_4_feature_importance.png")

print("\n  Top 10 Features by Permutation Importance:")
print(perm_imp.head(10).round(4).to_string())

  Saved: plot_4_feature_importance.png

  Top 10 Features by Permutation Importance:
ROA_Stability        0.1378
Loans_Assets         0.0189
MTBV                 0.0133
NPL_Loans_pct        0.0000
ROA                  0.0000
ROE                  0.0000
CostToIncomeRatio    0.0000
Formula_Fin_Lev      0.0000
TotalRevenue         0.0000
PE                   0.0000


# PART 8 — FINAL MODEL DIAGNOSTICS (XGBoost on Full Feature Set)


In [ ]:
xgb = XGBClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.05,
    scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
    use_label_encoder=False, eval_metric="logloss",
    random_state=42, verbosity=0
)

# Collect OOF predictions across folds for diagnostics
from sklearn.model_selection import cross_val_predict

y_pred_oof = cross_val_predict(xgb, X_scaled, y, cv=CV, method="predict")
y_prob_oof = cross_val_predict(xgb, X_scaled, y, cv=CV, method="predict_proba")[:, 1]

print(f"\n  OOF Accuracy : {accuracy_score(y, y_pred_oof):.4f}")
print(f"  OOF F1-Score : {f1_score(y, y_pred_oof):.4f}")
print(f"  OOF ROC-AUC  : {roc_auc_score(y, y_prob_oof):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("XGBoost Diagnostics — Out-of-Fold (Financial + NLP)", fontsize=13)

# Confusion matrix
cm = confusion_matrix(y, y_pred_oof)
disp = ConfusionMatrixDisplay(cm, display_labels=["No Controversy", "Controversy"])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix")

# ROC curve
RocCurveDisplay.from_predictions(y, y_prob_oof, ax=axes[1],
                                  color="#ED7D31", name="XGBoost (OOF)")
axes[1].plot([0,1],[0,1],"--", color="grey", alpha=0.5, label="Random")
axes[1].legend(fontsize=9)
axes[1].set_title("ROC Curve")

# Precision-Recall curve
PrecisionRecallDisplay.from_predictions(y, y_prob_oof, ax=axes[2],
                                         color="#5B9BD5", name="XGBoost (OOF)")
axes[2].set_title("Precision-Recall Curve")

plt.tight_layout()
plt.savefig("plot_5_xgb_diagnostics.png", dpi=150)
plt.close()
print("  Saved: plot_5_xgb_diagnostics.png")



  OOF Accuracy : 0.6333
  OOF F1-Score : 0.6667
  OOF ROC-AUC  : 0.7149
  Saved: plot_5_xgb_diagnostics.png


# PART 9 — SUMMARY TABLE


In [ ]:
print("\n[PART 9] Summary …")

summary_rows = []
for exp_name, res in [("Financial Only", res_fin), ("Financial + NLP", res_all)]:
    for model_name, scores in res.items():
        row = {"Experiment": exp_name, "Model": model_name}
        row.update(scores)
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).round(4)
print("\n" + "=" * 70)
print("  FINAL RESULTS TABLE (5-Fold Stratified CV)")
print("=" * 70)
print(summary_df.to_string(index=False))

# ── Save summary as CSV ─────────────────────────────────────────────────────
summary_df.to_csv("esg_results_summary.csv", index=False)
print("\n  Saved: esg_results_summary.csv")

# ── Feature importance CSV ──────────────────────────────────────────────────
imp_df = pd.DataFrame({
    "Feature"              : feature_names,
    "RF_Importance"        : [rf_imp[f] for f in feature_names],
    "Permutation_Importance": [perm_imp[f] for f in feature_names],
}).sort_values("Permutation_Importance", ascending=False)
imp_df.to_csv("esg_feature_importance.csv", index=False)
print("  Saved: esg_feature_importance.csv")

print("\n" + "=" * 70)
print("  PIPELINE COMPLETE  ✓")
print("=" * 70)



[PART 9] Summary …

  FINAL RESULTS TABLE (5-Fold Stratified CV)
     Experiment               Model  Accuracy  Precision  Recall     F1  ROC-AUC
 Financial Only Logistic Regression    0.4000     0.3533  0.3500 0.3467   0.4917
 Financial Only       Random Forest    0.6333     0.7200  0.6500 0.6548   0.6806
 Financial Only             XGBoost    0.6667     0.7667  0.6667 0.6762   0.7028
Financial + NLP Logistic Regression    0.4333     0.5533  0.4000 0.4267   0.3972
Financial + NLP       Random Forest    0.6667     0.7833  0.6667 0.6857   0.5861
Financial + NLP             XGBoost    0.6333     0.6667  0.6333 0.6395   0.6611

  Saved: esg_results_summary.csv
  Saved: esg_feature_importance.csv

  PIPELINE COMPLETE  ✓
